# Python 문법 파일별 5 · 17-1 인자 순서와 기본값 · `def`

**한 줄 요약** · 원본 함수가 왜 에러인지 확인해 고치고, 키워드 인자 · 기본값 함정 · 학습 함수 설계까지 정리

- **원본** · `1.Python문법/01_실습문제/17-1.Function Arguments_Q.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/01_Python문법` 전체 정리 4-1 (인자 순서와 기본값)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 원본 함수가 정의 단계에서 SyntaxError가 나는 이유를 말한다
2. 필수 인자 → 기본값 인자 순서로 함수를 정의한다
3. 위치 인자 · 키워드 인자(`이름=값`)로 같은 함수를 부른다
4. 기본값에 리스트를 쓰면 왜 위험한지 설명하고 `None`으로 고친다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **학습 함수에 epoch · 학습률 기본값을 둘 때** · `def train(model, epochs=10, lr=1e-3)` (3장)
- **층을 만들 때 옵션 일부만 바꿀 때** · `nn.Linear(192, 3, bias=False)` 키워드 인자 (2-1)
- **기록 리스트를 인자로 받을 때** · 기본값 `None` + 안에서 생성 (2-2)
- **남이 만든 함수의 에러 메시지를 읽을 때** · SyntaxError · TypeError 구분 (1-1 · 2-1)

## 학습 지도

<img src="images/pf5_map.png" alt="17-1 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 문제</td></tr>
<tr><td>1</td><td>인자 순서 고치기</td><td>원본 함수는 왜 에러가 나고, 어떻게 고치는가?</td></tr>
<tr><td>2</td><td>인자 넘기는 방법과 기본값 함정</td><td>이름으로 인자를 넘기는 법과 기본값 리스트의 함정은?</td></tr>
<tr><td>3</td><td>종합 실습</td><td>필수 인자 · 기본값 · <code>None</code> 기본값을 모두 쓰는 학습 함수를 만들려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

이 노트북은 라이브러리 없이 파이썬 기본 문법만 씀. **셀은 위에서부터 차례로 실행**(1-2의 `greet`를 2장에서 다시 씀).

*항목 → 설정*

- **라이브러리** · 없음
- **seed · 장치** · 필요 없음
- **데이터** · 원본 문제의 `greet` 함수 · 3장은 가짜 손실 값

**실행 흐름**

1. `sys` 모듈 불러오기
2. 파이썬 버전 출력
3. 3.10 이상인지 확인

In [1]:
# 실행 환경 확인: 이 노트북은 파이썬 기본 문법만 씀
# 왜 · 버전 차이로 출력이 달라지지 않는지 먼저 확인하기 위해
import sys  # 파이썬 버전 확인용

print("Python:", sys.version.split()[0])  # 3.10 이상이면 됨

Python: 3.14.6


### 원본 문제

- 아래 함수가 실행 시 **에러가 나는 이유**를 생각하고, **함수를 고쳐** 정상 실행하는 문제

```python
def greet(msg="Good morning!", name):
    return "Hi! " + name + ', ' + msg

print(greet("Kim"))
print(greet("Park", "How do you do?"))
```

**원본 힌트**

- 파이썬 함수 인자의 순서는 중요
- 함수를 정의할 때 가변 인자 · 기본값을 쓰면 활용도와 가독성이 높아짐
- 원본은 인자 종류 5가지를 정리한 글과 그림을 함께 소개

---
## 1. 인자 순서 고치기

> **이 장의 흐름**
>
> - 원본 함수를 그대로 정의해 에러를 확인하고
> - 필수 인자를 앞으로 옮겨 원본 호출 2개를 실행

<img src="images/py1_function.png" alt="함수: 인자 → 매개변수 → 처리 → return" width="763">

**파이프라인 위치** · ④ 모델 · ⑤ 학습 — 함수로 묶기

### 1-1. `A` 원본 함수의 에러 확인 · `SyntaxError` · `exec()` <sub>원본 문제 · 에러 이유</sub>

> 🎯 **원본 함수는 정의할 때 에러가 나는가, 부를 때 에러가 나는가?**

💡 **쉽게 말하면**

- 함수는 **주문서**.
- 꼭 적을 칸(필수)이 기본값 칸 **뒤**에 오면 파이썬이 주문서 자체를 거절함.

**실행 흐름**

1. 원본 함수를 문자열로 보관
2. exec로 정의만 해 보기
3. SyntaxError 확인

**먼저 예측** · 에러가 나는 순간은 정의할 때일까, 부를 때일까? — 답은 코드 실행 결과 아래에

In [2]:
# 원본 함수를 문자열로 두고 정의만 해 봄 (그대로 쓰면 이 셀이 아예 실행되지 않음)
# 왜 · SyntaxError를 셀 전체가 멈추지 않게 붙잡아 에러 내용을 보기 위해
broken = 'def greet(msg="Good morning!", name):\n    return "Hi! " + name + ", " + msg'  # 원본 함수
try:
    exec(broken)                  # 문자열을 코드로 정의
except SyntaxError as e:          # 정의 단계 문법 에러
    print("SyntaxError:", e.msg)  # 에러 이유만

SyntaxError: parameter without a default follows parameter with a default


**결과** · 예측 정답은 정의할 때 — 실행 전 문법 검사에서 걸림

- `SyntaxError: parameter without a default follows parameter with a default` — 기본값 인자 뒤에 필수 인자 불가
- 에러 위치는 `greet(...)` 호출이 아니라 `def` 줄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>SyntaxError</td><td><strong>정의 · 문법</strong> 단계</td><td>코드 모양이 규칙에 어긋날 때 (셀 전체가 실행 안 됨)</td></tr>
<tr><td>TypeError</td><td><strong>호출</strong> 단계</td><td>인자 개수 · 종류가 안 맞을 때 (2-1)</td></tr>
<tr><td><code>exec(문자열)</code></td><td>문자열을 코드로 실행</td><td>에러 나는 코드를 셀 안에서 보여줄 때만</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원본 함수를 셀에 그대로 쓰면 셀 전체가 실행되지 않아 아래 코드도 안 돎 → 여기서는 문자열 + `exec`로 보여줌

> ✅ **한 줄 정리** · 기본값 인자 뒤에 필수 인자를 두면 정의 단계에서 SyntaxError가 난다.

**딥러닝 코드에서는**

- 에러 종류부터 읽기 — SyntaxError면 코드 모양, TypeError면 넘긴 인자를 확인

### 1-2. `A` 정답 · 순서를 고쳐 원본 호출 실행 · `def` · `msg="Good morning!"` <sub>원본 문제 · 수정</sub>

> 🎯 **인자 순서를 어떻게 바꾸면 원본 호출 2개가 모두 실행되는가?**

💡 **쉽게 말하면**

- 필수 칸 `name`을 **앞**, 기본값 칸 `msg`를 **뒤**로.
- 안 적으면 기본값, 적으면 적은 값.

**실행 흐름**

1. name을 앞, msg를 뒤로
2. 원본 호출 2개 실행

**먼저 예측** · `greet("Kim")`의 출력은? — 답은 코드 실행 결과 아래에

In [3]:
# 수정: 필수 인자 name을 앞, 기본값 인자 msg를 뒤로
# 왜 · 기본값 없는 인자가 앞에 와야 한다는 규칙을 지켜 에러를 없애기 위해
def greet(name, msg="Good morning!"):  # def 함수명(필수, 기본값=값):
    return "Hi! " + name + ", " + msg  # 인사 문장 돌려줌


# 왜 · 기본값을 쓸 때와 덮어쓸 때의 결과를 나란히 비교하기 위해
print(greet("Kim"))                     # msg 생략 → 기본값
print(greet("Park", "How do you do?"))  # 위치 순서대로 전달

Hi! Kim, Good morning!
Hi! Park, How do you do?


**결과** · 예측 정답은 `Hi! Kim, Good morning!`

- `Hi! Kim, Good morning!` — 생략하면 기본값
- `Hi! Park, How do you do?` — 두 번째 인자가 msg로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>def f(a, b=1)</code></td><td>필수 → 기본값 (<strong>올바름</strong>)</td><td>기본 선택</td></tr>
<tr><td><code>def f(a=1, b)</code></td><td>기본값 → 필수 (<strong>SyntaxError</strong>)</td><td>쓰면 안 됨</td></tr>
<tr><td><code>def f(a=1, b=2)</code></td><td>모두 기본값</td><td>인자 없이도 부를 수 있게 할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 순서를 바꾸면 기존 호출 `greet("Park", "How do you do?")`의 의미도 바뀜 → 호출 쪽도 같이 확인

> ✅ **한 줄 정리** · 필수 인자를 앞, 기본값 인자를 뒤에 두면 생략한 인자는 기본값으로 채워진다.

**딥러닝 코드에서는**

- `def train(model, loader, epochs=10):` — 꼭 필요한 것 먼저, 옵션은 뒤

---
## 2. 인자 넘기는 방법과 기본값 함정

> **이 장의 흐름**
>
> - 원본 힌트: 인자 순서 · 기본값을 쓰면 가독성이 높아짐
> - 이름=값 호출 · 빠진 인자 에러 · 기본값 리스트 공유와 해결

<img src="images/pf5_default.png" alt="기본값 리스트는 호출마다 공유, None은 호출마다 새로" width="763">

**파이프라인 위치** · ⑤ 학습 — 학습 함수 옵션 · 기록

### 2-1. `A` 키워드 인자와 빠진 인자 · `이름=값` · `TypeError` <sub>원본 힌트 · 인자 종류</sub>

> 🎯 **`이름=값`으로 부르면 무엇이 달라지고, 필수 인자를 빠뜨리면 어떻게 되는가?**

💡 **쉽게 말하면**

- `이름=값`은 **칸 이름을 적어 채우기** → 순서 상관없음.
- 필수 칸을 비우면 부를 때 TypeError.

**실행 흐름**

1. 이름=값으로 순서 바꿔 호출
2. 기본값 일부만 바꾸기
3. 필수 인자 빠뜨리기

**먼저 예측** · `greet(msg="Hello", name="Lee")`는 누구에게 인사하나? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 키워드 인자는 순서와 상관없고 위치 인자 뒤에만 올 수 있음을 확인하기 위해
print(greet(msg="Hello", name="Lee"))  # 이름=값 → 순서 무관
print(greet("Choi", msg="Bye"))        # 위치 인자 다음 키워드 인자
try:
    greet()                            # 필수 인자 name 없음
except TypeError as e:                 # 인자가 모자람
    print("TypeError:", e)             # 에러 문장 출력

Hi! Lee, Hello
Hi! Choi, Bye
TypeError: greet() missing 1 required positional argument: 'name'


**결과** · 예측 정답은 `Lee` — 이름으로 짝지어짐

- `Hi! Lee, Hello` — 이름=값이면 순서 무관
- `Hi! Choi, Bye` — 위치 인자 뒤에 키워드 인자
- `TypeError: greet() missing 1 required positional argument: 'name'` — 필수 인자는 꼭 전달

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>위치 인자 <code>f(a, b)</code></td><td><strong>순서</strong>대로 전달</td><td>인자가 1 ~ 2개로 순서가 뻔할 때</td></tr>
<tr><td>키워드 인자 <code>f(b=1, a=2)</code></td><td><strong>이름</strong>으로 전달, 순서 무관</td><td>인자가 많거나 기본값 일부만 바꿀 때</td></tr>
<tr><td>섞어 쓰기 <code>f(1, b=2)</code></td><td>위치 먼저, 키워드 나중</td><td>필수는 위치로, 옵션은 이름으로</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `greet(name="Lee", "Hi")`처럼 키워드 인자 뒤에 위치 인자를 쓰면 SyntaxError

> ✅ **한 줄 정리** · `이름=값`으로 부르면 순서와 상관없이 칸을 채우고, 필수 인자를 빼면 TypeError가 난다.

**딥러닝 코드에서는**

- `nn.Linear(192, 3, bias=False)` — 기본값 인자를 이름으로 바꾸기

### 2-2. `A` 기본값 리스트 함정과 `None` · `box=[]` · `box=None` <sub>원본 힌트 · 기본값</sub>

> 🎯 **기본값으로 `[]`를 쓰면 왜 이전 호출의 값이 남는가?**

💡 **쉽게 말하면**

- 기본값 `[]`는 **함수를 만들 때 한 번만** 만들어짐.
- 그래서 모든 호출이 **같은 상자**를 같이 씀.

**실행 흐름**

1. 기본값 []로 정의
2. 두 번 호출
3. None으로 고치기
4. 다시 두 번 호출

**먼저 예측** · `add_item("b")`를 두 번째로 부르면? — 답은 코드 실행 결과 아래에

In [5]:
# 함정: 기본값 리스트는 정의할 때 한 번만 만들어져 호출마다 공유됨
# 왜 · 기본값 리스트가 호출 사이에 남는 버그를 직접 재현하기 위해
def add_item(item, box=[]):  # 위험한 기본값
    box.append(item)         # 같은 리스트에 계속 추가
    return box               # 그 리스트 반환


print("1번째:", add_item("a"))  # ['a']
print("2번째:", add_item("b"))  # a가 남아 있음


# 해결: None으로 두고 함수 안에서 새로 만들기
# 왜 · 호출마다 새 리스트를 만들어 기록이 섞이지 않게 하기 위해
def add_item_safe(item, box=None):  # 기본값은 None
    if box is None:                 # 넘겨받은 리스트가 없으면
        box = []                    # 호출마다 새 리스트
    box.append(item)                # 이번 호출의 리스트에 추가
    return box                      # 돌려줌


print("safe:", add_item_safe("a"), add_item_safe("b"))  # 서로 따로

1번째: ['a']
2번째: ['a', 'b']
safe: ['a'] ['b']


**결과** · 예측 정답은 `['a', 'b']` — 기본값 리스트가 공유됨

- 기본값 리스트가 호출마다 공유됨
  - `1번째: ['a']`
  - `2번째: ['a', 'b']`
- `safe: ['a'] ['b']` — `None`으로 두고 안에서 새로 만들면 해결

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>기본값 <code>[]</code> · <code>{}</code></td><td>정의할 때 <strong>한 번</strong> 생성 → 공유</td><td>쓰면 안 됨</td></tr>
<tr><td>기본값 <code>None</code> + 안에서 생성</td><td>호출마다 <strong>새로</strong> 생성</td><td>리스트 · 딕셔너리를 기본값으로 쓰고 싶을 때</td></tr>
<tr><td>기본값 숫자 · 문자열</td><td>바꿀 수 없는 값이라 안전</td><td><code>lr=1e-3</code> · <code>msg="..."</code></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `if not box:`로 확인하면 사용자가 넘긴 빈 리스트도 새로 바꿔 버림 → `is None`으로 확인

> ✅ **한 줄 정리** · 바뀌는 객체를 기본값으로 쓰지 말고 `None`으로 둔 뒤 함수 안에서 새로 만든다.

**딥러닝 코드에서는**

- `def train(..., history=None):` → `history = history or {}` 대신 `if history is None:`

---
## 3. 종합 실습 · 학습 함수 설계

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - 딥러닝 `train` 함수 모양을 가짜 계산으로 흉내 냄

<img src="images/pf5_practice.png" alt="train 함수: 필수 인자 · 기본값 · None 기록" width="763">

**파이프라인 위치** · ⑤ 학습

> 🎯 **`train(losses, epochs=3, lr=0.1, history=None)`을 만들어 여러 방식으로 호출하려면?**

💡 **쉽게 말하면**

- **필수**(데이터) → **옵션**(epochs · lr) → **기록**(None) 순서.
- 부를 때 옵션은 이름으로 바꿈.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>함수 정의</td><td>1-2 필수 → 기본값 순서</td></tr>
<tr><td>2</td><td>기록 None 처리</td><td>2-2 <code>None</code> + 안에서 생성</td></tr>
<tr><td>3</td><td>기본값으로 호출</td><td>1-2 생략 → 기본값</td></tr>
<tr><td>4</td><td>옵션 일부만 바꿔 호출</td><td>2-1 키워드 인자</td></tr>
</tbody></table>

**먼저 예측** · `train(losses)`를 두 번 불러도 기록이 섞이지 않을까? — 답은 코드 실행 결과 아래에

In [6]:
# 1 · 2단계: 필수 → 기본값 → None 기록 순서로 정의 (실제 학습 대신 가짜 계산)
# 왜 · 학습 함수의 인자 순서 규칙과 None 기본값을 한 함수에 모으기 위해
def train(losses, epochs=3, lr=0.1, history=None):          # 필수 · 옵션 · 기록
    if history is None:                                     # 기록을 안 넘기면
        history = []                                        # 호출마다 새 리스트
    for epoch in range(epochs):                             # epoch 수만큼
        history.append(round(losses[epoch] * (1 - lr), 3))  # 손실을 lr만큼 줄인 척
    return history                                          # 기록 반환


losses = [0.9, 0.6, 0.5, 0.4]  # 가짜 손실 값


# 3단계: 기본값으로 두 번 호출 (기록이 섞이지 않아야 함)
# 왜 · history=None 덕분에 두 번 불러도 결과가 같은지 확인하기 위해
print("기본값 1:", train(losses))  # epochs=3 · lr=0.1
print("기본값 2:", train(losses))  # 새 리스트라 같은 결과


# 4단계: 옵션 일부만 이름으로 바꾸기
# 왜 · 필요한 옵션만 이름으로 바꾸고 나머지는 기본값을 쓰기 위해
print("lr만:", train(losses, lr=0.5))             # epochs는 기본값
print("둘 다:", train(losses, epochs=2, lr=0.0))  # 순서 대신 이름

기본값 1: [0.81, 0.54, 0.45]
기본값 2: [0.81, 0.54, 0.45]
lr만: [0.45, 0.3, 0.25]
둘 다: [0.9, 0.6]


**결과** · 예측 정답은 섞이지 않음 — `None`이라 호출마다 새 리스트

- 두 번 불러도 같은 결과 (기록 공유 없음)
  - `기본값 1: [0.81, 0.54, 0.45]`
  - `기본값 2: [0.81, 0.54, 0.45]`
- `lr만: [0.45, 0.3, 0.25]` — `lr=0.5`만 바꾸고 epochs는 기본값 3
- `둘 다: [0.9, 0.6]` — epochs=2 · lr=0 → 원래 값 2개

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>losses</code> (필수)</td><td>없으면 학습 불가</td><td>맨 앞 · 위치 인자</td></tr>
<tr><td><code>epochs=3</code> · <code>lr=0.1</code></td><td>대부분 기본값으로 충분</td><td>바꿀 때만 이름으로</td></tr>
<tr><td><code>history=None</code></td><td>넘기면 이어서 기록 · 안 넘기면 새로</td><td>바뀌는 객체 옵션</td></tr>
<tr><td><code>history=[]</code> (쓰면 안 됨)</td><td>호출마다 기록이 쌓임</td><td>2-2 함정</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `epochs`가 `losses` 길이보다 크면 IndexError → 실제 학습에서는 데이터로더 길이와 맞춤

> ✅ **한 줄 정리** · 학습 함수는 필수 인자 → 기본값 옵션 → `None` 기록 순서로 만들고, 옵션은 이름으로 바꿔 부른다.

**딥러닝 코드에서는**

- `def fit(model, train_loader, val_loader=None, epochs=10, lr=1e-3):` — 같은 설계

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 원본 에러</td><td>기본값 인자 뒤 필수 인자 → 정의 단계 SyntaxError</td><td>1-1</td></tr>
<tr><td>2. 정답</td><td><code>def greet(name, msg=...)</code> · 원본 호출 2개 실행</td><td>1-2</td></tr>
<tr><td>3. 넘기는 방법</td><td>키워드 인자는 순서 무관 · 빠진 필수 인자는 TypeError</td><td>2-1</td></tr>
<tr><td>4. 기본값 함정</td><td><code>[]</code> 공유 → <code>None</code> + 안에서 생성</td><td>2-2</td></tr>
<tr><td>5. 학습 함수</td><td>필수 → 옵션 → None 기록 순서</td><td>3</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>인자 순서</td><td>필수 → 기본값</td><td><code>def train(model, epochs=10)</code></td></tr>
<tr><td><code>A</code></td><td>키워드 인자</td><td><code>이름=값</code> · 순서 무관</td><td><code>nn.Linear(192, 3, bias=False)</code></td></tr>
<tr><td><code>A</code></td><td>기본값 <code>None</code></td><td>바뀌는 객체는 안에서 생성</td><td><code>history=None</code></td></tr>
<tr><td><code>B</code></td><td>SyntaxError vs TypeError</td><td>정의 단계 vs 호출 단계</td><td>에러 메시지 읽기</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · `def f(a=1, b):`는 언제 어떤 에러가 나는가?
- **1-2** · 원본을 어떻게 고치는가?
- **2-1** · `greet()`를 부르면?
- **2-2** · 기본값 `[]`가 위험한 이유는?
- **3** · 학습 함수의 인자 순서는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `def area(w, h=1):`로 넓이 함수를 만들고 `area(3)` · `area(3, 4)` · `area(h=2, w=5)` 결과 출력
2. `add_item_safe("x", ["a"])`의 결과 출력

In [7]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `def power(x, n=2):`를 만들고 `power(3)` · `power(2, n=10)` 결과 말하기
2. `def log(msg, tags=[]):`를 안전하게 고치기
3. `greet("Lee", name="Kim")`은 어떻게 되는가?

In [8]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 정의할 때 SyntaxError
- **1-2** · `def greet(name, msg="Good morning!")`
- **2-1** · TypeError — 필수 인자 name 누락
- **2-2** · 정의할 때 한 번 만들어져 호출마다 공유
- **3** · 필수(데이터) → 옵션(기본값) → 기록(None)

#### 코드 문제 답

```python
def area(w, h=1):
    return w * h


def add_item_safe(item, box=None):
    if box is None:
        box = []
    box.append(item)
    return box


print(area(3), area(3, 4), area(h=2, w=5))  # 3 12 10
print(add_item_safe("x", ["a"]))  # ['a', 'x']
```

#### 변형 문제 답

*문제 → 답*

- **1** · `9` · `1024`
- **2** · `def log(msg, tags=None):` + `if tags is None: tags = []`
- **3** · TypeError — name에 값이 두 번 전달됨 (multiple values)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 기본값 <code>[]</code> 함수를 두 번 불러 결과가 쌓임</td><td>기본값이 호출마다 새로 만들어진다고 착각</td><td>기본값은 정의할 때 한 번만 생성</td><td>바뀌는 기본값이면 <code>None</code>으로 바꿨나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 함수 인자의 순서 규칙은 무엇인가요?**
> - 정의할 때는 필수 인자를 먼저, 기본값 인자를 뒤에 둬야 하며 어기면 정의 단계에서 SyntaxError가 납니다.
> - 부를 때는 위치 인자를 먼저 쓰고 키워드 인자를 뒤에 씁니다.

**Q2. 기본값으로 빈 리스트를 쓰면 왜 안 되나요?**
> - 기본값은 함수를 정의할 때 한 번만 만들어지므로 모든 호출이 같은 리스트를 공유합니다.
> - 그래서 이전 호출의 값이 남습니다.
> - 기본값을 `None`으로 두고 함수 안에서 새 리스트를 만들면 해결됩니다.

**Q3. 키워드 인자는 언제 쓰나요?**
> - 인자가 많거나 기본값 중 일부만 바꿀 때 씁니다.
> - `nn.Linear(192, 3, bias=False)`처럼 이름을 적으면 순서를 외우지 않아도 되고 코드만 봐도 뜻이 보입니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>def</code> · <code>return</code></td><td>Python 기본 문법</td><td>함수 정의 · 결과 돌려주기</td><td>1-2 <code>def greet(name, msg=...)</code></td></tr>
<tr><td>기본값 인자 <code>이름=값</code></td><td>Python 기본 문법</td><td>생략하면 기본값</td><td>1-2 <code>msg="Good morning!"</code></td></tr>
<tr><td><code>exec()</code> · <code>SyntaxError</code></td><td>Python 내장 함수 · 예외</td><td>문자열을 코드로 · 문법 에러</td><td>1-1 <code>exec(broken)</code></td></tr>
<tr><td>키워드 인자 · <code>TypeError</code></td><td>Python 기본 문법 · 예외</td><td>이름으로 전달 · 인자 개수 에러</td><td>2-1 <code>greet(msg="Hello", name="Lee")</code></td></tr>
<tr><td>기본값 <code>None</code> · <code>is None</code></td><td>Python 기본 문법</td><td>호출마다 새 객체</td><td>2-2 <code>def add_item_safe(item, box=None)</code></td></tr>
<tr><td><code>round()</code> · <code>range()</code></td><td>Python 내장 함수</td><td>반올림 · 반복 횟수</td><td>3 <code>round(losses[epoch] * (1 - lr), 3)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>필수 인자 → 기본값 인자 순서</td><td><code>def f(a=1, b):</code></td></tr>
<tr><td>옵션은 <code>이름=값</code>으로 호출</td><td>기본값으로 <code>[]</code> · <code>{}</code></td></tr>
<tr><td><code>if x is None:</code>으로 기본값 확인</td><td><code>if not x:</code>로 None과 빈 리스트를 섞기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>SyntaxError vs TypeError</td><td>정의 단계 vs 호출 단계</td><td>에러 이름부터 확인</td></tr>
<tr><td>위치 인자 vs 키워드 인자</td><td>순서 vs 이름</td><td>옵션은 이름으로</td></tr>
<tr><td>기본값 <code>[]</code> vs <code>None</code></td><td>공유 vs 호출마다 새로</td><td><code>None</code></td></tr>
<tr><td><code>is None</code> vs <code>not x</code></td><td>None만 vs 빈 값 모두</td><td>기본값 확인은 <code>is None</code></td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 다시 풀기 · 원본 함수를 직접 고쳐 쓰기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>3장 train에 <code>verbose=False</code> 옵션을 추가해 다시 만들기</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 6(24-1) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 1장 · 원본 문제(`17-1.Function Arguments_Q.ipynb`)의 에러와 수정
- 2장 · 힌트의 인자 종류 · 기본값을 키워드 인자 · 함정으로 확장
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>문제 · 에러 이유</td><td>원본 함수의 에러 확인</td><td>1-1</td></tr>
<tr><td>문제 · 함수 고치기</td><td>순서를 고쳐 원본 호출 실행</td><td>1-2</td></tr>
<tr><td>힌트 · 인자 순서 · 종류</td><td>키워드 인자와 빠진 인자</td><td>2-1</td></tr>
<tr><td>힌트 · 기본값 활용</td><td>기본값 리스트 함정과 None</td><td>2-2</td></tr>
<tr><td>힌트 · 인자 종류 그림</td><td>0장 원본 힌트 · 3장 학습 함수</td><td>0장 · 3</td></tr>
</tbody></table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 잘못된 정의의 SyntaxError 확인
2. 1-2 인자 순서를 고친 `greet()` 정의 · 호출
3. 블록마다 결과 출력

In [9]:
# ---------- 1-1 ----------
# 원본 함수를 문자열로 두고 정의만 해 봄 (그대로 쓰면 이 셀이 아예 실행되지 않음)
# 왜 · SyntaxError를 셀 전체가 멈추지 않게 붙잡아 에러 내용을 보기 위해
broken = 'def greet(msg="Good morning!", name):\n    return "Hi! " + name + ", " + msg'  # 원본 함수
try:
    exec(broken)                  # 문자열을 코드로 정의
except SyntaxError as e:          # 정의 단계 문법 에러
    print("SyntaxError:", e.msg)  # 에러 이유만


# ---------- 1-2 ----------
# 수정: 필수 인자 name을 앞, 기본값 인자 msg를 뒤로
# 왜 · 기본값 없는 인자가 앞에 와야 한다는 규칙을 지켜 에러를 없애기 위해
def greet(name, msg="Good morning!"):  # def 함수명(필수, 기본값=값):
    return "Hi! " + name + ", " + msg  # 인사 문장 돌려줌


# 왜 · 기본값을 쓸 때와 덮어쓸 때의 결과를 나란히 비교하기 위해
print(greet("Kim"))                     # msg 생략 → 기본값
print(greet("Park", "How do you do?"))  # 위치 순서대로 전달

SyntaxError: parameter without a default follows parameter with a default
Hi! Kim, Good morning!
Hi! Park, How do you do?


**실행 흐름**

1. 2-1 키워드 인자 호출
2. 2-2 기본값 리스트 함정과 `None` 해결
3. 블록마다 반환값 출력

In [10]:
# ---------- 2-1 ----------
# 왜 · 키워드 인자는 순서와 상관없고 위치 인자 뒤에만 올 수 있음을 확인하기 위해
print(greet(msg="Hello", name="Lee"))  # 이름=값 → 순서 무관
print(greet("Choi", msg="Bye"))        # 위치 인자 다음 키워드 인자
try:
    greet()                            # 필수 인자 name 없음
except TypeError as e:                 # 인자가 모자람
    print("TypeError:", e)             # 에러 문장 출력


# ---------- 2-2 ----------
# 함정: 기본값 리스트는 정의할 때 한 번만 만들어져 호출마다 공유됨
# 왜 · 기본값 리스트가 호출 사이에 남는 버그를 직접 재현하기 위해
def add_item(item, box=[]):  # 위험한 기본값
    box.append(item)         # 같은 리스트에 계속 추가
    return box               # 그 리스트 반환


print("1번째:", add_item("a"))  # ['a']
print("2번째:", add_item("b"))  # a가 남아 있음


# 해결: None으로 두고 함수 안에서 새로 만들기
# 왜 · 호출마다 새 리스트를 만들어 기록이 섞이지 않게 하기 위해
def add_item_safe(item, box=None):  # 기본값은 None
    if box is None:                 # 넘겨받은 리스트가 없으면
        box = []                    # 호출마다 새 리스트
    box.append(item)                # 이번 호출의 리스트에 추가
    return box                      # 돌려줌


print("safe:", add_item_safe("a"), add_item_safe("b"))  # 서로 따로

Hi! Lee, Hello
Hi! Choi, Bye
TypeError: greet() missing 1 required positional argument: 'name'
1번째: ['a']
2번째: ['a', 'b']
safe: ['a'] ['b']


**실행 흐름**

1. `train()` 정의 (필수 → 기본값 → `None`)
2. 기본값으로 두 번 호출
3. 옵션 일부만 이름으로 바꿔 호출

In [11]:
# ---------- 3. 종합 실습 ----------
# 1 · 2단계: 필수 → 기본값 → None 기록 순서로 정의 (실제 학습 대신 가짜 계산)
# 왜 · 학습 함수의 인자 순서 규칙과 None 기본값을 한 함수에 모으기 위해
def train(losses, epochs=3, lr=0.1, history=None):          # 필수 · 옵션 · 기록
    if history is None:                                     # 기록을 안 넘기면
        history = []                                        # 호출마다 새 리스트
    for epoch in range(epochs):                             # epoch 수만큼
        history.append(round(losses[epoch] * (1 - lr), 3))  # 손실을 lr만큼 줄인 척
    return history                                          # 기록 반환


losses = [0.9, 0.6, 0.5, 0.4]  # 가짜 손실 값


# 3단계: 기본값으로 두 번 호출 (기록이 섞이지 않아야 함)
# 왜 · history=None 덕분에 두 번 불러도 결과가 같은지 확인하기 위해
print("기본값 1:", train(losses))  # epochs=3 · lr=0.1
print("기본값 2:", train(losses))  # 새 리스트라 같은 결과


# 4단계: 옵션 일부만 이름으로 바꾸기
# 왜 · 필요한 옵션만 이름으로 바꾸고 나머지는 기본값을 쓰기 위해
print("lr만:", train(losses, lr=0.5))             # epochs는 기본값
print("둘 다:", train(losses, epochs=2, lr=0.0))  # 순서 대신 이름

기본값 1: [0.81, 0.54, 0.45]
기본값 2: [0.81, 0.54, 0.45]
lr만: [0.45, 0.3, 0.25]
둘 다: [0.9, 0.6]
